# mPES - Colab Launcher desde GitHub

Este notebook clona una copia ligera del repositorio en el almacenamiento local
de Colab y ejecuta la optimizacion desde `h1/`. Los resultados se conservan
en Google Drive.

Configura en la primera celda el repositorio y la rama. El clonado usa
`--depth 1 --single-branch` para reducir tiempo, espacio y trafico de red.
Para `ens_sprb` o `ens_accq`, la rama clonada debe incluir los tres modelos en
sus rutas canonicas dentro de `h1/ml/.../inputs/`.

Ejecuta todas las celdas en orden. Para ejecuciones largas, activa Background
execution en la sesion de Colab.

In [ ]:
"""Colab launcher for mPES Bayesian optimisation."""
# Mount Google Drive before cloning the repository.
# ==========================================================================
# MOUNT GOOGLE DRIVE
# ==========================================================================
# pyright: reportMissingImports=false
# pylint: disable=import-error,no-name-in-module
from google.colab import drive  # type: ignore[import-not-found]
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Configure a Colab optimisation from a shallow Git clone.
import os
import subprocess

REPOSITORY   = 'https://github.com/Maximiliano0/mPES_2026.git'
BRANCH       = 'new_uq'
WORKSPACE    = '/content/mPES'
H_DIR        = os.path.join(WORKSPACE, 'h1')
UTILS_DIR    = os.path.join(WORKSPACE, 'utils')
OUTPUT_ROOT  = '/content/drive/MyDrive/mPES/runs'
PKG          = 'ens_sprb'  # ql | dql | dqn | rdqn | ac | tr | ens_sprb | ens_accq
N_TRIALS     = 50
RESUME_DATE  = ''  # YYYY-MM-DD to resume, or '' for a new run
USE_GPU      = 0

valid_packages = ('ql', 'dql', 'dqn', 'rdqn', 'ac', 'tr', 'ens_sprb', 'ens_accq')
if PKG not in valid_packages:
    raise ValueError(f'Unsupported PKG: {PKG!r}')

os.environ.update({
    'DRIVE_DIR': OUTPUT_ROOT,
    'H_DIR': H_DIR,
    'REPO_DIR': WORKSPACE,
    'WORKSPACE_DIR': WORKSPACE,
    'PKG': PKG,
    'N_TRIALS': str(N_TRIALS),
    'RESUME_DATE': RESUME_DATE,
    'MPES_USE_GPU': str(USE_GPU),
    'MPES_MODEL_ROOT': '',
})
print(f'[INFO] [Celda 1] Configuración guardada: PKG={PKG!r}, N_TRIALS={N_TRIALS}, BRANCH={BRANCH!r}')

In [ ]:
# Shallow clone: only the selected branch and its current snapshot.
if os.path.isdir(WORKSPACE):
    print(f"[INFO] [Celda 2] Borrando workspace anterior en {WORKSPACE}...")
    subprocess.run(['rm', '-rf', WORKSPACE], check=True)

print(f"[INFO] [Celda 2] Clonando el repositorio rama '{BRANCH}'...")
subprocess.run([
    'git', 'clone', '--depth', '1', '--single-branch', '--branch', BRANCH,
    REPOSITORY, WORKSPACE,
], check=True)

print("[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...")
subprocess.run(
    ['bash', os.path.join(H_DIR, 'general', 'colab', 'setup_colab.sh')],
    check=True,
    env=os.environ.copy(),
)
print(f'[INFO] [Celda 2] Shallow clone y setup completados: {REPOSITORY}@{BRANCH}')

[INFO] [Celda 2] Borrando workspace anterior en /content/mPES...
[INFO] [Celda 2] Clonando el repositorio rama 'new_uq'...
[INFO] [Celda 2] Ejecutando setup_colab.sh (Instalación de dependencias)...


KeyboardInterrupt: 

In [ ]:
_script = '''
set -uo pipefail
cd "$WORKSPACE"
source /content/mpes_env.sh
bash "$H_DIR/general/colab/run_colab.sh" "$PKG" "$N_TRIALS" "$RESUME_DATE"
'''
print("[INFO] [Celda 3] Iniciando run_colab.sh... Observa los registros a continuación:")
_proc = subprocess.run(
    _script, shell=True, executable='/bin/bash', check=False, env=os.environ.copy(),
)
if _proc.returncode != 0:
    print(f"[ERROR] [Celda 3] run_colab.sh falló con código {_proc.returncode}")
    raise RuntimeError(f'run_colab.sh exited with code {_proc.returncode}')
print("[INFO] [Celda 3] ¡Ejecución de run_colab.sh completada con éxito!")